# A paper-structure NPS example: Pol II / RNA29

This offline example connects the RNA29 measurements **without TFIIB** in Table 1 of Muschielok et al., *Nature Methods* 5, 965–971 (2008), DOI `10.1038/nmeth.1259`, to the structure used for Figure 5. The paper cites PDB `1Y1W` (Kettenberger et al., PDB DOI `10.2210/pdb1Y1W/pdb`).

The observations used here are Rpb7-Cys150: 60.4 ± 0.6%, R0=65 Å; Rpb7-Cys94: 52.2 ± 1.1%, R0=66 Å; and Rpb7-S16C: 73.6 ± 0.7%, R0=62 Å. These are three separately labeled RNA29–Rpb7 donor/acceptor pairs, not simultaneous three-color measurements. Fractions, not percentage points, are passed to BFF; each R0 is the paper's pair-specific *isotropic* Förster radius. The paper's Rpb4-S73C observation is bimodal and is intentionally excluded.

This is a deliberately incomplete structural teaching approximation: each SG/OG is an **attachment-site surrogate**, not a dye center. The paper's 7 Å satellite linker, 7 Å spherical dye, available-volume steric restriction, and orientation-radius priors are not represented. Therefore neither candidate below is an actual paper fit, and this notebook does not claim the original posterior, uncertainty, or convergence.

The crystal's chain G is Rpb7 and chain D is Rpb4. Chain P contains only 10 nt, so the RNA29 5' donor is **ABSENT** from this structure. Chains T and N are DNA and are shown separately from the 3' RNA chain P.

In [ ]:
from pathlib import Path
import gzip
import hashlib
import io

import IMP.bff as bff
import matplotlib.pyplot as plt
import numpy as np
from Bio.PDB import PDBParser
from scipy.spatial import cKDTree

DATA = Path('data/1Y1W.pdb.gz')
raw = gzip.open(DATA, 'rb').read()
assert hashlib.sha256(raw).hexdigest() == '4225117bfc9f1dbb2c2edd278c7087cbf4acffe6af8654a74466726dbba9ca6f'
structure = PDBParser(QUIET=True).get_structure('1Y1W', io.StringIO(raw.decode('ascii')))
model = structure[0]
assert set(('G', 'D', 'P', 'T', 'N')).issubset({chain.id for chain in model})
assert len(list(model['P'].get_residues())) == 10
assert model['G'][150]['SG'].get_parent().resname == 'CYS'
assert model['G'][94]['SG'].get_parent().resname == 'CYS'
assert model['G'][16]['OG'].get_parent().resname == 'SER'
assert model['D'][73]['OG'].get_parent().resname == 'SER'
print('PDB 1Y1W verified; SHA256:', hashlib.sha256(raw).hexdigest())
print('chains:', {chain.id: len(list(chain.get_residues())) for chain in model})

In [ ]:
labels = ['Rpb7-Cys150', 'Rpb7-Cys94', 'Rpb7-S16C']
anchor_atoms = [('G', 150, 'SG'), ('G', 94, 'SG'), ('G', 16, 'OG')]
anchors = np.array([model[c][r][a].coord for c, r, a in anchor_atoms], dtype=float)
observed_e = np.array([0.604, 0.522, 0.736])
observed_sigma = np.array([0.006, 0.011, 0.007])
r0 = np.array([65.0, 66.0, 62.0])
target_distances = r0 * ((1.0 / observed_e) - 1.0) ** (1.0 / 6.0)
print('attachment-site surrogates (Å):')
for label, point in zip(labels, anchors): print(f'  {label}: {point}')
print('distance targets (Å):', np.round(target_distances, 6))

## Three spheres, two mirror solutions

The distances above use an **isotropic point-dye approximation** to the measured efficiencies. We solve the three-sphere intersection analytically, rather than seeding known coordinates. The two mirror solutions fit these three efficiencies equally; they are not draws from the paper's orientational/accessible-volume posterior, and the point clash screen below is not a posterior prior.

In [ ]:
def sphere_intersections(centers, radii):
    """Analytic intersections of three non-collinear distance spheres."""
    p0, p1, p2 = centers
    ex = p1 - p0
    d = np.linalg.norm(ex)
    assert d > 0
    ex /= d
    offset = p2 - p0
    i = np.dot(ex, offset)
    lateral = offset - i * ex
    j = np.linalg.norm(lateral)
    assert j > 0
    ey = lateral / j
    ez = np.cross(ex, ey)
    x = (radii[0]**2 - radii[1]**2 + d**2) / (2*d)
    y = (radii[0]**2 - radii[2]**2 + i**2 + j**2 - 2*i*x) / (2*j)
    z_squared = radii[0]**2 - x**2 - y**2
    assert z_squared > 0, 'the three measured spheres must intersect twice'
    midpoint = p0 + x*ex + y*ey
    return np.array([midpoint + np.sqrt(z_squared)*ez,
                     midpoint - np.sqrt(z_squared)*ez])

candidates = sphere_intersections(anchors, target_distances)
# Order for presentation only: a point inside the protein is candidate 1.
protein_heavy_atoms = np.array([
    atom.coord for chain in model if chain.id in set('ABCDEFGHIJKL')
    for atom in chain.get_atoms() if atom.element not in ('H', 'D')
])
nearest_to_protein = cKDTree(protein_heavy_atoms).query(candidates)[0]
candidates = candidates[np.argsort(nearest_to_protein)]
assert np.allclose(np.linalg.norm(candidates[:, None, :] - anchors, axis=2),
                   target_distances, atol=1e-9)
assert np.allclose(candidates[0], [82.93, 8.99, -20.77], atol=0.02)
assert np.allclose(candidates[1], [178.08, 19.40, -54.76], atol=0.02)
print('mirror candidates (Å):')
for i, candidate in enumerate(candidates, 1): print(f'  candidate {i}: {candidate}')

In [ ]:
def make_dye():
    dye = bff.NPSNetworkDye()
    dye.dep, dye.iso, dye.dist_conv = 0.0, True, False
    return dye

dyes = [make_dye() for _ in range(4)]
measurements = []
for i, (e, sigma, radius) in enumerate(zip(observed_e, observed_sigma, r0)):
    measurement = bff.NPSMeasurement()
    measurement.dye1, measurement.dye2 = i, 3
    measurement.fret_active, measurement.e_avg, measurement.e_err = True, float(e), float(sigma)
    measurement.r_iso6 = float(radius ** 6)
    measurements.append(measurement)

def config_for(candidate):
    return [list(point) + [0.0, 0.0] for point in np.vstack((anchors, candidate))]

predicted = []
log_likelihoods = []
for candidate in candidates:
    config = config_for(candidate)
    pred = np.array([bff.nps_network_fret_efficiency(config, dyes, m.dye1, m.dye2, m.r_iso6) for m in measurements])
    predicted.append(pred)
    log_likelihoods.append(bff.nps_network_log_likelihood(config, dyes, measurements))
predicted = np.array(predicted)
for i, pred in enumerate(predicted, 1):
    print(f'candidate {i} measured / BFF predicted E:', list(zip(np.round(observed_e, 4), np.round(pred, 4))))
    print(f'candidate {i} normalized BFF log L:', log_likelihoods[i - 1])
assert np.allclose(predicted, observed_e, atol=1e-10)
assert np.isfinite(log_likelihoods).all()

In [ ]:
nearest_heavy_atom = cKDTree(protein_heavy_atoms).query(candidates)[0]
print('crude nearest protein heavy-atom distances (Å):', np.round(nearest_heavy_atom, 3))
assert nearest_heavy_atom[0] < 2.0 and nearest_heavy_atom[1] > 20.0
print('Diagnostic only: candidate 1 clashes; candidate 2 does not under this crude point test.')
print('This does not establish real dye localization or reproduce the paper posterior.')

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
colors = {'D': '#dd8800', 'G': '#2277bb'}
for chain in model:
    # Coarse C-alpha traces are only structural context, not a rendered molecular surface.
    ca = np.array([res['CA'].coord for res in chain if 'CA' in res])
    if len(ca):
        ax.plot(ca[:, 0], ca[:, 1], ca[:, 2],
                color=colors.get(chain.id, '#999999'), linewidth=0.65,
                alpha=0.6 if chain.id in ('D', 'G') else 0.35)
for chain_id, color, label in [('P', '#cc33aa', "RNA P (3' segment, 10 nt)"),
                               ('T', '#33aa55', 'DNA T'), ('N', '#008844', 'DNA N')]:
    residues = [res for res in model[chain_id] if 'P' in res or "O5'" in res]
    # One phosphate per residue, except the terminal nucleotide without P.
    backbone = np.array([res['P'].coord if 'P' in res else res["O5'"].coord
                         for res in residues])
    assert len(backbone) == len(residues)
    ax.plot(backbone[:, 0], backbone[:, 1], backbone[:, 2],
            color=color, linewidth=2.0, label=label)
ax.scatter(anchors[:, 0], anchors[:, 1], anchors[:, 2], color='#000000',
           s=45, label='SG/OG attachment-site surrogates')
for label, point in zip(labels, anchors):
    ax.text(*point, '  ' + label.replace('Rpb7-', ''), fontsize=7)
ax.scatter(*candidates[0], color='#e41a1c', s=70,
           label='candidate 1 (point clash)')
ax.scatter(*candidates[1], color='#4daf4a', s=70,
           label='candidate 2 (no point clash)')
for i, point in enumerate(candidates, 1): ax.text(*point, f'  candidate {i}')
ax.set(xlabel='x (Å)', ylabel='y (Å)', zlabel='z (Å)',
       title='1Y1W structure and simplified NPS geometry')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()